# PolERo – open-weight LLM test (train fold 0 only, never test)
Few-shot **Qwen3-14B (AWQ 4-bit)** with vLLM on a Colab free **T4**. It classifies the 656 fold-0 train rows (`common.folds(...)[0][1]`) into the 9 evasion labels. Few-shot examples come only from folds 1–4.
Setup: Runtime → Change runtime type → **T4 GPU**; Colab secret `KAGGLE_API_TOKEN` with notebook access on; then Run all. Allow Drive when asked.
Output: `MyDrive/polero/llm/llm_fold0_<model>.csv` (`row_idx, datapointID, gold, pred`) plus the macro-F1 printed at the end. Compare against the v1 ensemble OOF on the same rows with `llm/compare_llm.py` in the repo.

In [ ]:
MODEL = "Qwen/Qwen3-14B-AWQ"   # fits a 16 GB T4; on Kaggle T4 x2 try "Qwen/Qwen3-30B-A3B-GPTQ-Int4" with TP=2
TP = 1                         # tensor parallel = number of GPUs
SHOTS_PER_CLASS = 2
N_ROWS = None                  # None = all 656 fold-0 rows; e.g. 100 for a quick smoke test
MAX_ANSWER_WORDS = 350         # long answers are cut to their start + end

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/polero/llm"; os.makedirs(OUT, exist_ok=True)
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!pip -q install -U kaggle six
!mkdir -p /content/polero/data
!kaggle datasets download lucazanotti/polero-data -p /content/polero --unzip -o
%cd /content/polero
!mv -f *.csv data/ 2>/dev/null; ls data

In [ ]:
# vLLM install takes ~3-5 min. Then ALWAYS do Runtime -> Restart session and Run all again.
# nvidia-cutlass-dsl: vLLM 0.34 imports cutlass during kernel warm-up; without it LLM() fails with
# No module named cutlass. The kernel only sees it after a restart (a .pth file adds it to sys.path).
!pip -q install vllm nvidia-cutlass-dsl

In [ ]:
import numpy as np, pandas as pd, re
from sklearn.metrics import f1_score, classification_report
from common import load, folds, EVASION_LABELS, EVASION_TO_CLARITY
tr, _ = load()                       # the test split is NOT used anywhere below
fs = folds(tr.evasion_label.values)
val_idx = fs[0][1]
if N_ROWS: val_idx = np.random.RandomState(0).permutation(val_idx)[:N_ROWS]
pool = tr.drop(index=fs[0][1])        # few-shot pool: folds 1-4 only

def cut(text, n=MAX_ANSWER_WORDS):
    w = str(text).split()
    return text if len(w) <= n else " ".join(w[: n // 2]) + " [...] " + " ".join(w[-n // 2:])

DEFS = '''You label how a Romanian president answers a journalist's question. Pick exactly ONE label:
- Explicit: the requested information is given directly and explicitly.
- Implicit: the information is given, but only indirectly; it has to be inferred.
- General: the answer stays vague or too general and avoids the specifics asked for.
- Partial/half-answer: the question asks several things (or one compound thing) and only part of it is answered.
- Dodging: the answer ignores the question and talks about something else.
- Deflection: the answer stays near the topic but shifts the focus elsewhere (to another issue, actor, or a criticism) instead of answering.
- Declining to answer: the speaker openly refuses or says they will not answer.
- Claims ignorance: the speaker says they do not know or lack the information.
- Clarification: the speaker asks for clarification or does not understand the question instead of answering.
Reply with the label only.'''

rng = np.random.RandomState(1)
shots = []
for lab in EVASION_LABELS:
    cand = pool[pool.evasion_label == lab]
    shots += [cand.loc[i] for i in rng.choice(cand.index, min(SHOTS_PER_CLASS, len(cand)), replace=False)]
rng.shuffle(shots)

def user_msg(q, a):
    return f"Question: {cut(q, 120)}\nAnswer: {cut(a)}\nLabel:"

def messages(q, a):
    m = [{"role": "system", "content": DEFS}]
    for s in shots:
        m += [{"role": "user", "content": user_msg(s.interview_question, s.interview_answer)},
              {"role": "assistant", "content": s.evasion_label}]
    return m + [{"role": "user", "content": user_msg(q, a)}]

print(len(val_idx), "rows to label,", len(shots), "few-shot examples")

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(MODEL, dtype="float16", tensor_parallel_size=TP, max_model_len=12288, enforce_eager=True,
          gpu_memory_utilization=0.92, enable_prefix_caching=True)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(messages(r.interview_question, r.interview_answer), tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)
           for r in tr.loc[val_idx].itertuples()]
print("longest prompt (tokens):", max(len(tok(p).input_ids) for p in prompts))
outs = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=12))

In [ ]:
def parse(t):
    t = t.strip().lower()
    for lab in sorted(EVASION_LABELS, key=len, reverse=True):
        if t.startswith(lab.lower()) or lab.lower() in t: return lab
    for key, lab in [("partial", "Partial/half-answer"), ("declin", "Declining to answer"),
                     ("ignor", "Claims ignorance"), ("clarif", "Clarification")]:
        if key in t: return lab
    return "Explicit"   # fallback: the majority class

raw = [o.outputs[0].text for o in outs]
pred = [parse(t) for t in raw]
gold = tr.loc[val_idx, "evasion_label"].tolist()
print("unparsed outputs:", sum(1 for t in raw if not any(l.lower() in t.lower() for l in EVASION_LABELS)))
res = pd.DataFrame({"row_idx": val_idx, "datapointID": tr.loc[val_idx, "datapointID"].values, "gold": gold, "pred": pred, "raw": raw})
path = f"{OUT}/llm_fold0_{MODEL.split('/')[-1]}.csv"; res.to_csv(path, index=False); print("saved", path)
print("evasion macro-F1:", round(f1_score(gold, pred, average="macro"), 4))
print("clarity macro-F1 (mapped):", round(f1_score([EVASION_TO_CLARITY[g] for g in gold], [EVASION_TO_CLARITY[p] for p in pred], average="macro"), 4))
print(classification_report(gold, pred, labels=EVASION_LABELS, zero_division=0))